In [5]:
from galfits import images as IM
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
import matplotlib.pyplot as plt
from galfits import gsutils  # assumed to be available for image normalization

band = "K_cut"

# Define file name and load the image
fname = '../data/tmass{0}'.format(band)
img = IM.image(fname+'.fits', unit='cR')
hdu = fits.open(fname+'.fits')
header = hdu[0].header
data_org = hdu[0].data

# Define the output file for the sky-subtracted image
ss_imgname = '../data/tmass_{0}_ssimg.fits'.format(band)

# Perform sky subtraction with a 3rd order polynomial
maskplus = img.sky_subtraction(polyfit=True, order=3, filepath=ss_imgname,
                               psfFWHM=3, nsigma=3.0, npixels=5,
                               max_grow_size=100, addgrow=0)

# Calculate sigma-clipped statistics for visualization
sky_mean, sky_median, sky_std = sigma_clipped_stats(data_org, sigma=3.0, maxiters=5)
immin = 1 * sky_std
immax = 5 * sky_std

# Visualize the original image, sky-subtracted image, and the subtracted sky
fig = plt.figure(figsize=(18,6))
ax1 = fig.add_subplot(131)
ax1.imshow(gsutils.normimg(data_org, immin, immax, sky=sky_median, frac=0.6),
           cmap='seismic', origin='lower', vmin=-1, vmax=1, interpolation='nearest')
ax2 = fig.add_subplot(132)
ax2.imshow(gsutils.normimg(data_org - img.data.data, immin, immax, sky=sky_median, frac=0.6),
           cmap='seismic', origin='lower', vmin=-1, vmax=1, interpolation='nearest')
ax3 = fig.add_subplot(133)
ax3.imshow(gsutils.normimg(img.data.data, immin, immax, frac=0.6),
           cmap='seismic', origin='lower', vmin=-1, vmax=1, interpolation='nearest')
plt.show()

finish mask


TypeError: Improper input: func input vector length N=10 must not exceed func output vector length M=0

In [ ]:
# Define cutout radius in arcseconds
cutr = 2.5  # unit: arcsec

# Extract the image cutout around the target (ra, dec)
imcut, cp = img.img_cut(ra, dec, cutr)
header = hdu[0].header

# Calculate the sigma image for the cutout
sigimg = img.cal_sigma_image(200, gain=1.)
img.cut_sigma_image = sigimg[cp[2]:cp[3], cp[0]:cp[1]]

# Generate the cut mask image using the specified parameters
img.cut_mask_image = img.generate_cutmask(psfFWHM, data=imcut,
                                           nsigma=2.0, npixels=150,
                                           nlevels=32, contrast=0.001,
                                           deblend=False, growmethod=True,
                                           max_grow_size=15, addgrow=addgrow)

# Write the cut images to a FITS file
outname = './cutimg/tmass_{0}_cut.fits'.format(band)
img.write_imcut(ra, dec, outname, header=header)

# Visualize the cutout with the mask overlay
fig = plt.figure(figsize=(8,8))
sky_mean, sky_median, sky_std = sigma_clipped_stats(imcut, sigma=3.0, maxiters=5)
sky_median = 0.0
immin = 5 * sky_std
immax = imcut.max()
frac = 0.4
plt.imshow(gsutils.normimg(imcut, immin, immax, sky=sky_median, frac=frac),
           cmap='seismic', origin='lower', vmin=-1, vmax=1, interpolation='nearest')
plt.imshow(mask0.astype(float), cmap='Blues', alpha=0.5 * mask0.astype(float),
           origin='lower', interpolation='nearest')
plt.savefig(resupath + '{0}_{1}_cutout.png'.format(tagname, band))
plt.show()